In [1]:
# Install required libraries
!pip install transformers datasets torch accelerate -q
print("✅ All libraries installed!")


✅ All libraries installed!


In [2]:
# Check GPU is working
import torch
print(f"GPU Available: {torch.cuda.is_available()}")
print(f"GPU Name: {torch.cuda.get_device_name(0)}")
print(f"Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

GPU Available: True
GPU Name: Tesla T4
Memory: 15.6 GB


In [3]:
from datasets import load_dataset

# Load Stanford Alpaca instruction dataset
dataset = load_dataset("tatsu-lab/alpaca")
print(f"Total examples: {len(dataset['train'])}")
print(f"\nSample example:")
print(dataset['train'][0])

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:124: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md:   0%|          | 0.00/7.47k [00:00<?, ?B/s]

data/train-00000-of-00001-a09b74b3ef9c3b(…):   0%|          | 0.00/24.2M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/52002 [00:00<?, ? examples/s]

Total examples: 52002

Sample example:
{'instruction': 'Give three tips for staying healthy.', 'input': '', 'output': '1.Eat a balanced diet and make sure to include plenty of fruits and vegetables. \n2. Exercise regularly to keep your body active and strong. \n3. Get enough sleep and maintain a consistent sleep schedule.', 'text': 'Below is an instruction that describes a task. Write a response that appropriately completes the request.\n\n### Instruction:\nGive three tips for staying healthy.\n\n### Response:\n1.Eat a balanced diet and make sure to include plenty of fruits and vegetables. \n2. Exercise regularly to keep your body active and strong. \n3. Get enough sleep and maintain a consistent sleep schedule.'}


In [4]:
from transformers import GPT2Tokenizer

# Load GPT-2 tokenizer
tokenizer = GPT2Tokenizer.from_pretrained("gpt2")
tokenizer.pad_token = tokenizer.eos_token

def format_example(example):
    """Convert Alpaca example to instruction format."""
    if example["input"]:
        prompt = f"### Instruction:\n{example['instruction']}\n\n### Input:\n{example['input']}\n\n### Response:\n{example['output']}"
    else:
        prompt = f"### Instruction:\n{example['instruction']}\n\n### Response:\n{example['output']}"
    return prompt

def filter_and_prepare(dataset, max_samples=5000, max_length=512):
    """Filter low quality examples and tokenize."""
    examples = []
    skipped = 0

    for item in dataset['train']:
        # Quality filters
        if len(item['output']) < 10:        # skip too short
            skipped += 1
            continue
        if len(item['output']) > 1000:      # skip too long
            skipped += 1
            continue
        if len(item['instruction']) < 5:    # skip bad instructions
            skipped += 1
            continue

        text = format_example(item)
        tokens = tokenizer(
            text,
            truncation=True,
            max_length=max_length,
            padding="max_length",
            return_tensors="pt"
        )

        examples.append({
            "input_ids": tokens["input_ids"].squeeze(),
            "attention_mask": tokens["attention_mask"].squeeze(),
        })

        if len(examples) >= max_samples:
            break

    print(f"✅ Prepared {len(examples)} examples")
    print(f"⏭️  Skipped {skipped} low quality examples")
    print(f"\nSample formatted prompt:")
    print(format_example(dataset['train'][0])[:300])
    return examples

# Run the pipeline
prepared_data = filter_and_prepare(dataset)

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/1.04M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

✅ Prepared 5000 examples
⏭️  Skipped 375 low quality examples

Sample formatted prompt:
### Instruction:
Give three tips for staying healthy.

### Response:
1.Eat a balanced diet and make sure to include plenty of fruits and vegetables. 
2. Exercise regularly to keep your body active and strong. 
3. Get enough sleep and maintain a consistent sleep schedule.


In [5]:
import torch
from torch.utils.data import Dataset, DataLoader

class AlpacaDataset(Dataset):
    def __init__(self, examples):
        self.examples = examples

    def __len__(self):
        return len(self.examples)

    def __getitem__(self, idx):
        return self.examples[idx]

def create_dataloaders(examples, train_split=0.9, batch_size=4):
    """Split into train/val and create dataloaders."""
    split = int(len(examples) * train_split)
    train_data = examples[:split]
    val_data = examples[split:]

    train_dataset = AlpacaDataset(train_data)
    val_dataset = AlpacaDataset(val_data)

    train_loader = DataLoader(
        train_dataset,
        batch_size=batch_size,
        shuffle=True
    )
    val_loader = DataLoader(
        val_dataset,
        batch_size=batch_size,
        shuffle=False
    )

    print(f"✅ Train batches: {len(train_loader)}")
    print(f"✅ Val batches:   {len(val_loader)}")
    print(f"✅ Batch size:    {batch_size}")
    print(f"✅ Train examples: {len(train_data)}")
    print(f"✅ Val examples:   {len(val_data)}")
    return train_loader, val_loader

train_loader, val_loader = create_dataloaders(prepared_data)

✅ Train batches: 1125
✅ Val batches:   125
✅ Batch size:    4
✅ Train examples: 4500
✅ Val examples:   500


In [6]:
from transformers import GPT2LMHeadModel
import math

# Load pretrained GPT-2
model = GPT2LMHeadModel.from_pretrained("gpt2")
model = model.cuda()  # Move to GPU

# Optimizer
optimizer = torch.optim.AdamW(model.parameters(), lr=5e-5, weight_decay=0.01)

# Cosine LR scheduler — implemented manually
def get_lr(step, warmup_steps=100, total_steps=1000, max_lr=5e-5, min_lr=1e-6):
    if step < warmup_steps:
        return max_lr * step / warmup_steps
    progress = (step - warmup_steps) / (total_steps - warmup_steps)
    return min_lr + 0.5 * (max_lr - min_lr) * (1 + math.cos(math.pi * progress))

# Mixed precision scaler
scaler = torch.cuda.amp.GradScaler()

print(f"✅ Model loaded!")
print(f"✅ Parameters: {sum(p.numel() for p in model.parameters()):,}")
print(f"✅ Device: {next(model.parameters()).device}")

config.json:   0%|          | 0.00/665 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/548M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

✅ Model loaded!
✅ Parameters: 124,439,808
✅ Device: cuda:0


/tmp/ipykernel_1854/2611037456.py:19: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = torch.cuda.amp.GradScaler()


In [7]:
EPOCHS = 3
LOG_EVERY = 50
SAVE_EVERY = 200
best_val_loss = float("inf")
step = 0
train_losses = []
val_losses = []

print("🚀 Starting GPT-2 Fine-Tuning...\n")

for epoch in range(EPOCHS):
    model.train()
    epoch_loss = 0

    for batch_idx, batch in enumerate(train_loader):
        # Move to GPU
        input_ids = batch["input_ids"].cuda()
        attention_mask = batch["attention_mask"].cuda()

        # Update learning rate manually
        lr = get_lr(step)
        for param_group in optimizer.param_groups:
            param_group["lr"] = lr

        # Mixed precision forward pass
        with torch.cuda.amp.autocast():
            outputs = model(
                input_ids=input_ids,
                attention_mask=attention_mask,
                labels=input_ids
            )
            loss = outputs.loss

        # Backward pass
        optimizer.zero_grad()
        scaler.scale(loss).backward()

        # Gradient clipping
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)

        scaler.step(optimizer)
        scaler.update()

        epoch_loss += loss.item()
        step += 1

        if step % LOG_EVERY == 0:
            avg_loss = epoch_loss / (batch_idx + 1)
            perplexity = math.exp(avg_loss)
            print(f"Epoch {epoch+1} | Step {step} | Loss: {avg_loss:.4f} | PPL: {perplexity:.2f} | LR: {lr:.2e}")
            train_losses.append(avg_loss)

    # Validation
    model.eval()
    val_loss = 0
    with torch.no_grad():
        for batch in val_loader:
            input_ids = batch["input_ids"].cuda()
            attention_mask = batch["attention_mask"].cuda()
            with torch.cuda.amp.autocast():
                outputs = model(
                    input_ids=input_ids,
                    attention_mask=attention_mask,
                    labels=input_ids
                )
            val_loss += outputs.loss.item()

    avg_val_loss = val_loss / len(val_loader)
    val_perplexity = math.exp(avg_val_loss)
    val_losses.append(avg_val_loss)

    print(f"\n{'='*60}")
    print(f"Epoch {epoch+1} Complete!")
    print(f"Val Loss: {avg_val_loss:.4f} | Val Perplexity: {val_perplexity:.2f}")

    # Save best model
    if avg_val_loss < best_val_loss:
        best_val_loss = avg_val_loss
        model.save_pretrained("/content/gpt2-alpaca-best")
        tokenizer.save_pretrained("/content/gpt2-alpaca-best")
        print(f"✅ Best model saved!")
    print(f"{'='*60}\n")

print("🎉 Training Complete!")

🚀 Starting GPT-2 Fine-Tuning...



/tmp/ipykernel_1854/3509506866.py:26: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast():
[transformers] `loss_type=None` was set in the config but it is unrecognized. Using the default loss: `ForCausalLMLoss`.


Epoch 1 | Step 50 | Loss: 3.2250 | PPL: 25.15 | LR: 2.45e-05
Epoch 1 | Step 100 | Loss: 1.8163 | PPL: 6.15 | LR: 4.95e-05
Epoch 1 | Step 150 | Loss: 1.3368 | PPL: 3.81 | LR: 4.96e-05
Epoch 1 | Step 200 | Loss: 1.0990 | PPL: 3.00 | LR: 4.86e-05
Epoch 1 | Step 250 | Loss: 0.9418 | PPL: 2.56 | LR: 4.68e-05
Epoch 1 | Step 300 | Loss: 0.8419 | PPL: 2.32 | LR: 4.43e-05
Epoch 1 | Step 350 | Loss: 0.7743 | PPL: 2.17 | LR: 4.13e-05
Epoch 1 | Step 400 | Loss: 0.7210 | PPL: 2.06 | LR: 3.78e-05
Epoch 1 | Step 450 | Loss: 0.6809 | PPL: 1.98 | LR: 3.40e-05
Epoch 1 | Step 500 | Loss: 0.6480 | PPL: 1.91 | LR: 2.98e-05
Epoch 1 | Step 550 | Loss: 0.6226 | PPL: 1.86 | LR: 2.56e-05
Epoch 1 | Step 600 | Loss: 0.5978 | PPL: 1.82 | LR: 2.13e-05
Epoch 1 | Step 650 | Loss: 0.5770 | PPL: 1.78 | LR: 1.72e-05
Epoch 1 | Step 700 | Loss: 0.5622 | PPL: 1.75 | LR: 1.33e-05
Epoch 1 | Step 750 | Loss: 0.5484 | PPL: 1.73 | LR: 9.82e-06
Epoch 1 | Step 800 | Loss: 0.5363 | PPL: 1.71 | LR: 6.79e-06
Epoch 1 | Step 850 | Los

/tmp/ipykernel_1854/3509506866.py:61: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast():



Epoch 1 Complete!
Val Loss: 0.3318 | Val Perplexity: 1.39


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

✅ Best model saved!

Epoch 2 | Step 1150 | Loss: 0.3040 | PPL: 1.36 | LR: 4.24e-06
Epoch 2 | Step 1200 | Loss: 0.3075 | PPL: 1.36 | LR: 6.68e-06
Epoch 2 | Step 1250 | Loss: 0.3036 | PPL: 1.35 | LR: 9.69e-06
Epoch 2 | Step 1300 | Loss: 0.3103 | PPL: 1.36 | LR: 1.32e-05
Epoch 2 | Step 1350 | Loss: 0.3131 | PPL: 1.37 | LR: 1.70e-05
Epoch 2 | Step 1400 | Loss: 0.3124 | PPL: 1.37 | LR: 2.12e-05
Epoch 2 | Step 1450 | Loss: 0.3179 | PPL: 1.37 | LR: 2.54e-05
Epoch 2 | Step 1500 | Loss: 0.3172 | PPL: 1.37 | LR: 2.97e-05
Epoch 2 | Step 1550 | Loss: 0.3179 | PPL: 1.37 | LR: 3.38e-05
Epoch 2 | Step 1600 | Loss: 0.3165 | PPL: 1.37 | LR: 3.77e-05
Epoch 2 | Step 1650 | Loss: 0.3178 | PPL: 1.37 | LR: 4.12e-05
Epoch 2 | Step 1700 | Loss: 0.3159 | PPL: 1.37 | LR: 4.42e-05
Epoch 2 | Step 1750 | Loss: 0.3180 | PPL: 1.37 | LR: 4.67e-05
Epoch 2 | Step 1800 | Loss: 0.3203 | PPL: 1.38 | LR: 4.85e-05
Epoch 2 | Step 1850 | Loss: 0.3196 | PPL: 1.38 | LR: 4.96e-05
Epoch 2 | Step 1900 | Loss: 0.3198 | PPL: 1.38 | 

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

✅ Best model saved!

Epoch 3 | Step 2300 | Loss: 0.2943 | PPL: 1.34 | LR: 2.98e-05
Epoch 3 | Step 2350 | Loss: 0.2889 | PPL: 1.34 | LR: 2.56e-05
Epoch 3 | Step 2400 | Loss: 0.2919 | PPL: 1.34 | LR: 2.13e-05
Epoch 3 | Step 2450 | Loss: 0.2964 | PPL: 1.35 | LR: 1.72e-05
Epoch 3 | Step 2500 | Loss: 0.2900 | PPL: 1.34 | LR: 1.33e-05
Epoch 3 | Step 2550 | Loss: 0.2871 | PPL: 1.33 | LR: 9.82e-06
Epoch 3 | Step 2600 | Loss: 0.2862 | PPL: 1.33 | LR: 6.79e-06
Epoch 3 | Step 2650 | Loss: 0.2869 | PPL: 1.33 | LR: 4.33e-06
Epoch 3 | Step 2700 | Loss: 0.2860 | PPL: 1.33 | LR: 2.51e-06
Epoch 3 | Step 2750 | Loss: 0.2853 | PPL: 1.33 | LR: 1.39e-06
Epoch 3 | Step 2800 | Loss: 0.2857 | PPL: 1.33 | LR: 1.00e-06
Epoch 3 | Step 2850 | Loss: 0.2857 | PPL: 1.33 | LR: 1.36e-06
Epoch 3 | Step 2900 | Loss: 0.2842 | PPL: 1.33 | LR: 2.45e-06
Epoch 3 | Step 2950 | Loss: 0.2840 | PPL: 1.33 | LR: 4.24e-06
Epoch 3 | Step 3000 | Loss: 0.2842 | PPL: 1.33 | LR: 6.68e-06
Epoch 3 | Step 3050 | Loss: 0.2842 | PPL: 1.33 | 

In [9]:
import torch
import torch.nn.functional as F

def greedy_decode(model, tokenizer, prompt, max_new_tokens=100):
    """Always pick the highest probability token."""
    inputs = tokenizer(prompt, return_tensors="pt").to("cuda")
    input_len = inputs["input_ids"].shape[1]

    generated = inputs["input_ids"]
    for _ in range(max_new_tokens):
        with torch.no_grad():
            outputs = model(generated)
            logits = outputs.logits[:, -1, :]
        next_token = torch.argmax(logits, dim=-1, keepdim=True)
        generated = torch.cat([generated, next_token], dim=1)
        if next_token.item() == tokenizer.eos_token_id:
            break
    return tokenizer.decode(generated[0][input_len:], skip_special_tokens=True)


def temperature_decode(model, tokenizer, prompt, max_new_tokens=100, temperature=0.7):
    """Scale logits by temperature before sampling."""
    inputs = tokenizer(prompt, return_tensors="pt").to("cuda")
    input_len = inputs["input_ids"].shape[1]

    generated = inputs["input_ids"]
    for _ in range(max_new_tokens):
        with torch.no_grad():
            outputs = model(generated)
            logits = outputs.logits[:, -1, :] / temperature
        probs = F.softmax(logits, dim=-1)
        next_token = torch.multinomial(probs, num_samples=1)
        generated = torch.cat([generated, next_token], dim=1)
        if next_token.item() == tokenizer.eos_token_id:
            break
    return tokenizer.decode(generated[0][input_len:], skip_special_tokens=True)


def topk_decode(model, tokenizer, prompt, max_new_tokens=100, k=50):
    """Only sample from top K tokens."""
    inputs = tokenizer(prompt, return_tensors="pt").to("cuda")
    input_len = inputs["input_ids"].shape[1]

    generated = inputs["input_ids"]
    for _ in range(max_new_tokens):
        with torch.no_grad():
            outputs = model(generated)
            logits = outputs.logits[:, -1, :]
        # Zero out everything outside top K
        top_k_logits, top_k_indices = torch.topk(logits, k)
        filtered = torch.full_like(logits, float("-inf"))
        filtered.scatter_(1, top_k_indices, top_k_logits)
        probs = F.softmax(filtered, dim=-1)
        next_token = torch.multinomial(probs, num_samples=1)
        generated = torch.cat([generated, next_token], dim=1)
        if next_token.item() == tokenizer.eos_token_id:
            break
    return tokenizer.decode(generated[0][input_len:], skip_special_tokens=True)


def topp_decode(model, tokenizer, prompt, max_new_tokens=100, p=0.9):
    """Nucleus sampling — sample from smallest set of tokens covering p probability."""
    inputs = tokenizer(prompt, return_tensors="pt").to("cuda")
    input_len = inputs["input_ids"].shape[1]

    generated = inputs["input_ids"]
    for _ in range(max_new_tokens):
        with torch.no_grad():
            outputs = model(generated)
            logits = outputs.logits[:, -1, :]
        probs = F.softmax(logits, dim=-1)
        sorted_probs, sorted_indices = torch.sort(probs, descending=True)
        cumulative_probs = torch.cumsum(sorted_probs, dim=-1)
        # Remove tokens once cumulative prob exceeds p
        sorted_probs[cumulative_probs - sorted_probs > p] = 0
        sorted_probs /= sorted_probs.sum()
        next_token = sorted_indices[0][torch.multinomial(sorted_probs, num_samples=1)]
        next_token = next_token.unsqueeze(0).unsqueeze(0)
        generated = torch.cat([generated, next_token], dim=1)
        if next_token.item() == tokenizer.eos_token_id:
            break
    return tokenizer.decode(generated[0][input_len:], skip_special_tokens=True)


# Test all decoding methods
print("Loading best model...")
from transformers import GPT2LMHeadModel, GPT2Tokenizer
model = GPT2LMHeadModel.from_pretrained("/content/gpt2-alpaca-best").cuda()
tokenizer = GPT2Tokenizer.from_pretrained("/content/gpt2-alpaca-best")

prompt = "### Instruction:\nExplain what machine learning is.\n\n### Response:\n"

print("\n" + "="*60)
print("GREEDY DECODING:")
print(greedy_decode(model, tokenizer, prompt))

print("\n" + "="*60)
print("TEMPERATURE (0.7):")
print(temperature_decode(model, tokenizer, prompt, temperature=0.7))

print("\n" + "="*60)
print("TOP-K (k=50):")
print(topk_decode(model, tokenizer, prompt, k=50))

print("\n" + "="*60)
print("TOP-P (p=0.9):")
print(topp_decode(model, tokenizer, prompt, p=0.9))

Loading best model...


Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]


GREEDY DECODING:
Machine learning is a type of artificial intelligence that uses data to learn from data and generate predictions. It is used to improve customer service, automate processes, and automate processes.

TEMPERATURE (0.7):
Machine learning is an approach to understanding how machines learn from data. It uses machine learning techniques to identify patterns, predict patterns, and optimize algorithms. It then uses those patterns to learn from data and make predictions by analyzing it.

TOP-K (k=50):
Machine learning is an AI-driven approach that seeks to collect data on items, classify them into groups, and apply them to complex tasks. It involves finding patterns in the data, analyzing patterns to identify patterns and make predictions based on data analysis.

TOP-P (p=0.9):


RuntimeError: Tensors must have same number of dimensions: got 2 and 4

In [11]:
def topp_decode_fixed(model, tokenizer, prompt, max_new_tokens=100, p=0.9):
    """Fixed nucleus sampling."""
    inputs = tokenizer(prompt, return_tensors="pt").to("cuda")
    input_len = inputs["input_ids"].shape[1]

    generated = inputs["input_ids"]
    for _ in range(max_new_tokens):
        with torch.no_grad():
            outputs = model(generated)
            logits = outputs.logits[:, -1, :]  # (1, vocab_size)

        probs = F.softmax(logits, dim=-1).squeeze(0)  # (vocab_size,)
        sorted_probs, sorted_indices = torch.sort(probs, descending=True)
        cumulative_probs = torch.cumsum(sorted_probs, dim=0)

        # Remove tokens once cumulative prob exceeds p
        sorted_probs[cumulative_probs - sorted_probs > p] = 0
        sorted_probs /= sorted_probs.sum()

        sampled = torch.multinomial(sorted_probs, num_samples=1)
        next_token = sorted_indices[sampled].unsqueeze(0).unsqueeze(0)
        generated = torch.cat([generated, next_token], dim=1)

        if next_token.item() == tokenizer.eos_token_id:
            break

    return tokenizer.decode(generated[0][input_len:], skip_special_tokens=True)

# Test fixed Top-P
print("TOP-P (p=0.9) FIXED:")
print("="*60)
print(topp_decode_fixed(model, tokenizer, prompt, p=0.9))

print("\nTOP-P (p=0.7) — more focused:")
print("="*60)
print(topp_decode_fixed(model, tokenizer, prompt, p=0.7))

TOP-P (p=0.9) FIXED:


RuntimeError: Tensors must have same number of dimensions: got 2 and 3

In [12]:
def topp_decode_v2(model, tokenizer, prompt, max_new_tokens=100, p=0.9):
    """Clean nucleus sampling — fixed dimensions."""
    inputs = tokenizer(prompt, return_tensors="pt").to("cuda")
    input_len = inputs["input_ids"].shape[1]
    generated = inputs["input_ids"]

    for _ in range(max_new_tokens):
        with torch.no_grad():
            outputs = model(generated)
            logits = outputs.logits[0, -1, :]  # shape: (vocab_size,)

        probs = F.softmax(logits, dim=-1)
        sorted_probs, sorted_indices = torch.sort(probs, descending=True)
        cumsum = torch.cumsum(sorted_probs, dim=0)

        # Keep only tokens within top-p
        keep = (cumsum - sorted_probs) < p
        sorted_probs[~keep] = 0.0
        sorted_probs /= sorted_probs.sum()

        # Sample one token
        sampled_idx = torch.multinomial(sorted_probs, num_samples=1)
        next_token_id = sorted_indices[sampled_idx]

        # Reshape to (1,1) for concatenation
        next_token_id = next_token_id.view(1, 1)
        generated = torch.cat([generated, next_token_id], dim=1)

        if next_token_id.item() == tokenizer.eos_token_id:
            break

    return tokenizer.decode(generated[0][input_len:], skip_special_tokens=True)

# Test it
print("TOP-P (p=0.9):")
print("="*60)
print(topp_decode_v2(model, tokenizer, prompt, p=0.9))

print("\nTOP-P (p=0.7):")
print("="*60)
print(topp_decode_v2(model, tokenizer, prompt, p=0.7))

TOP-P (p=0.9):
Machine learning is a kind of artificial intelligence which uses data from a machine learning model to predict an action or consequence of an action or consequence. It is a branch of artificial intelligence that combines the intuitive, natural-like abilities of a computer or AI system. It helps machines to capture images and patterns, while providing insights into a user's preferences.

TOP-P (p=0.7):
Machine learning is a type of artificial intelligence (AI) which uses algorithms to analyze data to produce predictions, predict outcomes, and then use that data to develop new strategies. AI algorithms use data to create models, predict outcomes, and then use that data to generate predictions.


In [13]:
# Save the full notebook summary
print("="*60)
print("TRAINING SUMMARY")
print("="*60)
print(f"Base Model        : GPT-2 (124M parameters)")
print(f"Dataset           : Stanford Alpaca (5000 examples)")
print(f"Epochs            : 3")
print(f"Final Train Loss  : ~0.284")
print(f"Final Val Loss    : 0.3296")
print(f"Final Perplexity  : 1.39")
print(f"Decoding Methods  : Greedy, Temperature, Top-K, Top-P")
print("="*60)

# Download the model to your computer
from google.colab import files
import shutil

# Zip the model
shutil.make_archive("/content/gpt2-alpaca-best", "zip", "/content/gpt2-alpaca-best")

# Download it
files.download("/content/gpt2-alpaca-best.zip")
print("✅ Model downloading...")

TRAINING SUMMARY
Base Model        : GPT-2 (124M parameters)
Dataset           : Stanford Alpaca (5000 examples)
Epochs            : 3
Final Train Loss  : ~0.284
Final Val Loss    : 0.3296
Final Perplexity  : 1.39
Decoding Methods  : Greedy, Temperature, Top-K, Top-P


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

✅ Model downloading...


In [14]:
from google.colab import drive
drive.mount('/content/drive')

import shutil
shutil.copy("/content/gpt2-alpaca-best.zip", "/content/drive/MyDrive/gpt2-alpaca-best.zip")
print("✅ Model saved to Google Drive!")

Mounted at /content/drive
✅ Model saved to Google Drive!
